# C4 — An independent specificity axis from experimental PWMs (roadmap F1–F3)

**Why.** C1–C3 show that ΔΔG of binding cannot be learned from ddPCA data for SH3 domains, so the
atlas "rewiring" flag rests on zero-shot ESM-2 (B3). Both atlas axes therefore come from ESM, and
the thesis cannot test whether folding and binding effects really separate. This program tries
the one route the data allows: learn **binding specificity** from **experimental** phage-display
profiles (154 PWMs for human SH3 domains, Teyra et al. 2017) and use it as a second axis.

**What it does.**
1. Summarises each PWM as an **information-weighted amino-acid preference profile** (20 numbers).
   PWMs have different widths and no shared peptide frame, so position-by-position comparison is
   not possible; the profile is alignment-free.
2. Learns profile ← domain sequence (mean ESM-2 embedding), by ridge regression in centred
   log-ratio space, and tests it **leaving out whole clusters of related domains** (≥40 % identity),
   against two baselines: the average profile, and the profile of the most similar training domain.
3. **Pre-specified decision rule.** The axis counts as validated only if (a) the median centred
   correlation between predicted and observed profile has a 95 % bootstrap CI above zero **and**
   (b) the model beats the average profile on a sign test (p < 0.05).
4. If validated, scores every atlas variant: **rewiring = Jensen–Shannon distance between the
   predicted profiles of the wild-type and mutant domain**, then asks whether rewiring concentrates
   at the peptide interface and whether it is independent of predicted folding damage.

If the rule fails, the scores are still written but labelled *not validated*, and no rewiring claim
is made. A clear negative is reportable, just like the binding result.

**Inputs:** `input/datasets/SH3_PWM.zip`, `HumanSH3TableS2.xlsx` (upload these two), A0 atlas, A2
embeddings/structure, B1/B3 predictions. **Runtime:** about 10 min, **T4 GPU** (ESM passes for 6,884
mutant domains). **Outputs:** `pwm_profiles.csv`, `specificity_cv.csv`, `specificity_decision.json`,
`atlas_rewiring.parquet`, `RESULTS_C4.md`. Explanation: `docs/C4_specificity_axis.md`.

In [ ]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
os.system('pip -q install biopython openpyxl')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'C4', 'Independent specificity axis (PWMs)', resume=RESUME)
if C.DEV == 'cpu':
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.')
print(C.env_info())

In [ ]:
import zipfile, re, torch
from Bio import Align
from scipy.spatial.distance import jensenshannon
from scipy.stats import mannwhitneyu
from sklearn.linear_model import Ridge
R.verify_inputs(['input/datasets/SH3_PWM.zip', 'input/datasets/HumanSH3TableS2.xlsx'])
AAS = list('ACDEFGHIKLMNPQRSTVWY')
Z = zipfile.ZipFile(f'{R.IN}/datasets/SH3_PWM.zip'); pwm = {}
for n in Z.namelist():
    if not n.endswith('.txt') or '__MACOSX' in n: continue
    rows = {p[0]: [float(x) for x in p[1:]] for p in (l.split() for l in Z.read(n).decode().splitlines()) if p and p[0] in AAS and len(p) > 1}
    if len(rows) == 20:
        M_ = np.array([rows[a] for a in AAS]); pwm[os.path.basename(n)[:-4]] = M_ / M_.sum(0, keepdims=True)   # columns sum to 20 as shipped
raw = pd.read_excel(f'{R.IN}/datasets/HumanSH3TableS2.xlsx', header=None)
h = raw.index[raw.apply(lambda r: r.astype(str).str.strip().eq('Domain ID').any(), axis=1)][0]
S2 = pd.read_excel(f'{R.IN}/datasets/HumanSH3TableS2.xlsx', skiprows=h); S2.columns = [str(c).strip() for c in S2.columns]
S2 = S2.dropna(subset=['Domain ID', 'Domain Sequence']); S2['did'] = S2['Domain ID'].astype(str).str.replace('*', '', regex=False).str.strip()
seq_of = dict(zip(S2.did, S2['Domain Sequence'].astype(str).str.strip()))
def profile(P):
    ic = np.log2(20) + (P * np.log2(np.clip(P, 1e-12, 1))).sum(0)
    return (P * ic).sum(1) / ic.sum()
recs, unmatched = [], []
for name, P in pwm.items():
    did = re.sub(r'-\d+$', '', name).replace('_', '/')
    if did in seq_of: recs.append(dict(pwm=name, domain=did, seq=seq_of[did], width=P.shape[1], **dict(zip(AAS, profile(P)))))
    else: unmatched.append(name)
PR = pd.DataFrame(recs); PR.to_csv(R.out('pwm_profiles.csv'), index=False)
DOM = PR.groupby(['domain', 'seq'])[AAS].mean().reset_index()          # domains with two PWMs: average profile
print(f'{len(pwm)} PWMs, {len(PR)} matched to a domain sequence, {DOM.domain.nunique()} domains; unmatched: {unmatched}')

## 1. Features, clusters of related domains, and the cross-validated test

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
tok = AutoTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
esm = AutoModelForMaskedLM.from_pretrained('facebook/esm2_t33_650M_UR50D').to(C.DEV).eval()
if C.DEV == 'cuda': esm = esm.half()
@torch.no_grad()
def mean_embed(seq):      # the same representation as A2: last hidden layer, mean over residues
    enc = tok(seq, return_tensors='pt').to(C.DEV)
    return esm(**enc, output_hidden_states=True).hidden_states[-1][0, 1:len(seq) + 1].float().mean(0).cpu().numpy()
XP = np.stack([mean_embed(s) for s in DOM.seq])
clr = lambda P: np.log(np.clip(P, 1e-6, None)) - np.log(np.clip(P, 1e-6, None)).mean(1, keepdims=True)
soft = lambda Y: np.exp(Y) / np.exp(Y).sum(1, keepdims=True)
Y = clr(DOM[AAS].values)
idal = Align.PairwiseAligner(open_gap_score=-11, extend_gap_score=-1); idal.match_score = 1; idal.mismatch_score = 0
n = len(DOM); ID = np.eye(n)
for i in range(n):
    for j in range(i + 1, n):
        ID[i, j] = ID[j, i] = idal.score(DOM.seq[i], DOM.seq[j]) / min(len(DOM.seq[i]), len(DOM.seq[j]))
lab = np.arange(n)                                                      # single-linkage clusters at >=40% identity
for i in range(n):
    for j in range(n):
        if ID[i, j] >= 0.40: lab[lab == lab[j]] = lab[i]
print(f'{n} domains in {len(np.unique(lab))} clusters (>=40% identity)')
rows = []
for c in np.unique(lab):
    te = np.where(lab == c)[0]; tr = np.where(lab != c)[0]
    mu, sd = XP[tr].mean(0), XP[tr].std(0) + 1e-6; ybar = Y[tr].mean(0)
    alpha = max([1, 10, 100, 1000], key=lambda a: np.mean([np.corrcoef((Ridge(alpha=a).fit((XP[np.setdiff1d(tr, [k])] - mu) / sd, Y[np.setdiff1d(tr, [k])])
                                                                           .predict(((XP[k] - mu) / sd)[None])[0] - ybar), Y[k] - ybar)[0, 1]
                                                               for k in tr[::4]]))
    pred = Ridge(alpha=alpha).fit((XP[tr] - mu) / sd, Y[tr]).predict((XP[te] - mu) / sd)
    for k, t in enumerate(te):
        nn_ = tr[np.argmax(ID[t, tr])]
        rows.append(dict(domain=DOM.domain[t], cluster=int(c), alpha=alpha, nearest_identity=float(ID[t, nn_]),
                         r_model=float(np.corrcoef(pred[k] - ybar, Y[t] - ybar)[0, 1]),
                         r_nearest=float(np.corrcoef(Y[nn_] - ybar, Y[t] - ybar)[0, 1]),
                         js_model=float(jensenshannon(soft(pred[k][None])[0], DOM[AAS].values[t], base=2)),
                         js_nearest=float(jensenshannon(DOM[AAS].values[nn_], DOM[AAS].values[t], base=2)),
                         js_mean=float(jensenshannon(soft(ybar[None])[0], DOM[AAS].values[t], base=2))))
CV = pd.DataFrame(rows); CV.to_csv(R.out('specificity_cv.csv'), index=False)
bs = [np.median(np.random.default_rng(b).choice(CV.r_model, len(CV))) for b in range(2000)]
R_MED, (R_LO, R_HI) = float(CV.r_model.median()), np.percentile(bs, [2.5, 97.5])
WIN_MEAN = int((CV.js_model < CV.js_mean).sum()); WIN_NN = int((CV.js_model < CV.js_nearest).sum())
P_MEAN = C.sign_test_two_sided(WIN_MEAN, len(CV)) if WIN_MEAN > len(CV) / 2 else 1.0
VALID = bool(R_LO > 0 and P_MEAN < 0.05)
print(f'median centred r {R_MED:+.3f} [{R_LO:+.3f}, {R_HI:+.3f}] | beats the average profile on {WIN_MEAN}/{len(CV)} (p {P_MEAN:.2g}) '
      f'| beats nearest neighbour on {WIN_NN}/{len(CV)} | VALIDATED: {VALID}')
json.dump(dict(validated=VALID, median_centred_r=R_MED, ci=[float(R_LO), float(R_HI)], wins_vs_mean=WIN_MEAN,
               wins_vs_nearest=WIN_NN, domains=len(CV), sign_test_p=P_MEAN,
               rule='CI of median centred r above 0 AND sign test vs average profile p < 0.05'),
          open(R.out('specificity_decision.json'), 'w'), indent=1)

## 2. Score every atlas variant: rewiring = distance between predicted wild-type and mutant profiles

In [ ]:
mu, sd = XP.mean(0), XP.std(0) + 1e-6
alpha = int(CV.alpha.mode().iat[0]); final = Ridge(alpha=alpha).fit((XP - mu) / sd, Y)
MASTER = pd.read_csv(R.res('A0', 'SH3_cancer_master.tsv'), sep='\t', usecols=['variant_id', 'wt_domain_seq', 'mut_domain_seq'])
A = pd.read_parquet(R.res('B3', 'atlas_predictions.parquet'))[['variant_id', 'domain_key', 'gene', 'pos', 'ref_pos', 'pred_ddG_fold',
                                                              'fold_lo', 'esm_score', 'mechanism']].merge(MASTER, on='variant_id')
if R.done('atlas_rewiring.parquet'):
    A = pd.read_parquet(R.out('atlas_rewiring.parquet')); print('loaded from this folder')
else:
    wt_e = {s: mean_embed(s) for s in A.wt_domain_seq.unique()}; t0 = time.time()
    mut_e = np.stack([mean_embed(s) for s in A.mut_domain_seq]); print(f'mutant embeddings {(time.time() - t0) / 60:.1f} min')
    Pw = soft(final.predict((np.stack([wt_e[s] for s in A.wt_domain_seq]) - mu) / sd)); Pm = soft(final.predict((mut_e - mu) / sd))
    A['rewiring'] = [jensenshannon(a, b, base=2) for a, b in zip(Pw, Pm)]
    A.drop(columns=['wt_domain_seq', 'mut_domain_seq']).to_parquet(R.out('atlas_rewiring.parquet'), index=False)
del esm; torch.cuda.empty_cache()
STR = pd.read_csv(R.res('A2', 'structure_features.csv')); ref = STR[STR.domain_key == 'GRB2-SH3'].set_index('pos')
A['interface'] = A.ref_pos.map(ref.dist_peptide) < 5.0; A['buried'] = A.ref_pos.map(ref.rsasa) < 0.25
a_ = A.dropna(subset=['ref_pos'])
U = mannwhitneyu(a_[a_.interface].rewiring, a_[~a_.interface].rewiring, alternative='greater')
TOP = A.rewiring >= A.rewiring.quantile(0.9); DEST = A.fold_lo >= 1.0
TWO = pd.crosstab(np.where(DEST, 'destabilising', 'not destabilising'), np.where(TOP, 'top-10% rewiring', 'other'))
print(TWO); print(f'interface vs rest: median rewiring {a_[a_.interface].rewiring.median():.4f} vs {a_[~a_.interface].rewiring.median():.4f}, p {U.pvalue:.2g}')

## Results

In [ ]:
for k, v in [('pwms', len(pwm)), ('pwms_matched', len(PR)), ('domains', len(DOM)), ('clusters', int(len(np.unique(lab)))),
             ('median_centred_r', R_MED), ('median_centred_r_ci_lo', float(R_LO)), ('median_centred_r_ci_hi', float(R_HI)),
             ('wins_vs_average_profile', WIN_MEAN), ('wins_vs_nearest_neighbour', WIN_NN), ('sign_test_p', P_MEAN), ('validated', VALID),
             ('rho_rewiring_vs_predicted_fold', C.spearman(A.rewiring, A.pred_ddG_fold)),
             ('rho_rewiring_vs_esm_score', C.spearman(A.rewiring, A.esm_score)),
             ('rewiring_interface_median', float(a_[a_.interface].rewiring.median())),
             ('rewiring_other_median', float(a_[~a_.interface].rewiring.median())), ('interface_test_p', float(U.pvalue)),
             ('n_top_rewiring_not_destabilising', int((TOP & ~DEST).sum())), ('n_both_axes', int((TOP & DEST).sum()))]:
    R.metric(k, v)
R.section('Cross-validated test (whole clusters of related domains held out)', 'centred r: correlation of predicted and observed '
          'profile after removing the training average (the shared proline preference). js: Jensen–Shannon distance, lower is better.',
          CV.describe().loc[['count', 'mean', '50%', 'min', 'max']].reset_index(), '{:.3f}')
R.section('Decision', '```\n' + open(R.out('specificity_decision.json')).read() + '\n```')
R.section('Two-axis map of the atlas', ('**Validated axis.** ' if VALID else '**NOT validated — shown for completeness; no rewiring '
          'claim is made.** ') + f"Rewiring vs predicted folding damage ρ {R.metrics['rho_rewiring_vs_predicted_fold']:+.3f}; vs zero-shot "
          f"ESM ρ {R.metrics['rho_rewiring_vs_esm_score']:+.3f}. Interface positions: median rewiring "
          f"{R.metrics['rewiring_interface_median']:.4f} vs {R.metrics['rewiring_other_median']:.4f} elsewhere (one-sided p {U.pvalue:.2g}).",
          TWO.reset_index().rename(columns={'row_0': 'folding axis'}))
R.finish([f"{len(PR)} of {len(pwm)} PWMs linked to {len(DOM)} domain sequences; tested by holding out {len(np.unique(lab))} clusters of related domains.",
          f"Predicted vs observed specificity: median centred r {R_MED:+.3f} [95% CI {R_LO:+.3f}, {R_HI:+.3f}]; beats the average profile on "
          f"{WIN_MEAN}/{len(CV)} domains and the nearest-neighbour profile on {WIN_NN}/{len(CV)}.",
          f"Pre-specified rule: {'VALIDATED' if VALID else 'NOT validated'}.",
          (f"Rewiring is {'higher' if U.pvalue < 0.05 else 'not higher'} at the peptide interface (p {U.pvalue:.2g}) and correlates "
           f"ρ {R.metrics['rho_rewiring_vs_predicted_fold']:+.3f} with predicted folding damage."
           if VALID else 'Without a validated axis, the fold/rewiring separation stays untested; this is reported as a data limit.')])